# 05j-d — Trainable topology decoder micro-canary

This notebook tests the narrow hypothesis authorized by 05j-c: whether a small nonlinear decoder can exploit the verified multiscale tree context. The 48 train-only pairs are partitioned into 36 fit pairs and 12 family-stratified internal-calibration pairs. The pre-existing 05i-c normalizer stays frozen; all new PCA and topology-design transforms use fit only. Development is evaluated only after each checkpoint is frozen. Held-out data, rollout and full training remain sealed.

## 1. Coherent checkout and GPU runtime

In [ ]:
import os, subprocess, sys
from pathlib import Path
WORKSPACE = Path('/kaggle/working/hayflow_workspace')
ELM_REPO = WORKSPACE / 'elmneuron'
if not ELM_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pandas', 'pyarrow', 'pyyaml'], check=True)
sys.path.insert(0, str(ELM_REPO))
REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Revision:', REVISION)

In [ ]:
import h5py, json, numpy as np, pandas as pd, pyarrow, torch, yaml
assert torch.cuda.is_available(), 'Attiva una GPU Kaggle prima di eseguire 05j-d.'
print({'torch': torch.__version__, 'cuda': torch.cuda.get_device_name(0)})

## 2. Immutable inputs and exact 05j-c provenance

In [ ]:
import hashlib, shutil, zipfile
from src.hayflow_model.hines_state_normalization_repair import EXPECTED_05H_INDEX_SHA256
from src.hayflow_model.hines_netcon_semantic_repair import EXPECTED_05I_INDEX_SHA256
from src.hayflow_model.hines_synaptic_domain_repair import EXPECTED_05IB_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_recheck import EXPECTED_05IC_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_revision import EXPECTED_05J_INDEX_SHA256
from src.hayflow_model.hines_spatial_support_revision import EXPECTED_05JB_INDEX_SHA256
from src.hayflow_model.hines_trainable_topology_canary import EXPECTED_05JC_INDEX_SHA256
INPUT_ROOT = Path('/kaggle/input')
def extract_zip_safely(source, destination):
    source, destination = Path(source), Path(destination)
    marker = destination / '.source_size'; stamp = str(source.stat().st_size)
    if marker.is_file() and marker.read_text().strip() == stamp: return destination
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True); root = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert target == root or root in target.parents, member.filename
        archive.extractall(destination)
    marker.write_text(stamp); return destination
def first_existing(candidates, message):
    found = next((Path(p).resolve() for p in candidates if Path(p).exists()), None)
    assert found is not None, message
    return found
def artifact_index_matches(path, expected_sha256):
    if expected_sha256 is None: return True
    path = Path(path)
    try:
        if path.is_file():
            with zipfile.ZipFile(path) as archive:
                members = [name for name in archive.namelist() if name.replace('\\', '/').endswith('artifact_index.json')]
                return len(members) == 1 and hashlib.sha256(archive.read(members[0])).hexdigest() == expected_sha256
        index = path / 'artifact_index.json'
        return index.is_file() and hashlib.sha256(index.read_bytes()).hexdigest() == expected_sha256
    except (OSError, zipfile.BadZipFile): return False
def artifact_source(env_name, zip_name, marker, preferred_token, message, expected_index_sha256=None):
    explicit = [Path(os.environ[env_name]).expanduser()] if os.environ.get(env_name) else []
    archives = list(INPUT_ROOT.rglob(zip_name)); extracted = [p.parent for p in INPUT_ROOT.rglob(marker)]
    stem = Path(zip_name).stem.lower(); slug = stem.replace('_', '-')
    exact = [p for p in extracted if p.name.lower() == stem or any(part.lower() in {stem, slug} for part in p.parts[-3:])]
    preferred = [p for p in extracted if preferred_token in str(p).lower()]
    candidates, seen = [], set()
    for candidate in explicit + archives + exact + preferred + extracted:
        key = str(candidate.resolve()) if candidate.exists() else str(candidate)
        if key not in seen: candidates.append(candidate); seen.add(key)
    valid = [p for p in candidates if p.exists() and artifact_index_matches(p, expected_index_sha256)]
    assert valid, f'{message} Candidati non compatibili: {[str(p) for p in candidates]}'
    return valid[0].resolve()
topup_candidates = ([Path(os.environ['HAYFLOW_TOPUP_V3']).expanduser()] if os.environ.get('HAYFLOW_TOPUP_V3') else []) + list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip')) + [p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE = first_existing(topup_candidates, 'Top-up BAP v3 non trovato.')
TOPUP_ROOT = extract_zip_safely(TOPUP_SOURCE, '/kaggle/working/hayflow05jd_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifests = list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json')); assert len(manifests) == 1, manifests
COMPOSITE_MANIFEST = manifests[0]
base_candidates = ([Path(os.environ['HAYFLOW_BASE_DATASET']).expanduser()] if os.environ.get('HAYFLOW_BASE_DATASET') else []) + [p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()] + [p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE = first_existing(base_candidates, 'Dataset base targeted v1.1 non trovato.')
CHECKPOINT_05B_SOURCE = artifact_source('HAYFLOW_05B_ARTIFACT', 'hayflow_hines_canary_v2.zip', 'canary_models.pt', 'canary', 'Artefatto 05b non trovato.')
if CHECKPOINT_05B_SOURCE.name == 'checkpoints': CHECKPOINT_05B_SOURCE = CHECKPOINT_05B_SOURCE.parent
ARTIFACT_05C_SOURCE = artifact_source('HAYFLOW_05C_ARTIFACT', 'hayflow_hines_causal_isolation.zip', 'checkpoint_forensics.json', 'causal', 'Artefatto 05c non trovato.')
ARTIFACT_05D_SOURCE = artifact_source('HAYFLOW_05D_ARTIFACT', 'hayflow_hines_residual_conditioning.zip', 'free_residual_report.json', 'residual', 'Artefatto 05d non trovato.')
ARTIFACT_05E_SOURCE = artifact_source('HAYFLOW_05E_ARTIFACT', 'hayflow_hines_segment_capacity.zip', 'capacity_probe_report.json', 'capacity', 'Artefatto 05e non trovato.')
ARTIFACT_05F_SOURCE = artifact_source('HAYFLOW_05F_ARTIFACT', 'hayflow_hines_segment_micro_canary.zip', 'micro_canary_report.json', 'micro', 'Artefatto 05f non trovato.')
ARTIFACT_05G_SOURCE = artifact_source('HAYFLOW_05G_ARTIFACT', 'hayflow_hines_optimization_audit.zip', 'optimization_support.json', 'optimization', 'Artefatto 05g non trovato.')
ARTIFACT_05H_SOURCE = artifact_source('HAYFLOW_05H_ARTIFACT', 'hayflow_hines_representation_forensics.zip', 'representation_forensics_config.json', 'representation', 'Artefatto 05h non trovato.', EXPECTED_05H_INDEX_SHA256)
ARTIFACT_05I_SOURCE = artifact_source('HAYFLOW_05I_ARTIFACT', 'hayflow_hines_state_normalization_repair.zip', 'state_normalization_repair_config.json', 'state-normalization', 'Artefatto 05i non trovato.', EXPECTED_05I_INDEX_SHA256)
ARTIFACT_05IB_SOURCE = artifact_source('HAYFLOW_05IB_ARTIFACT', 'hayflow_hines_netcon_semantic_state_repair.zip', 'netcon_semantic_repair_config.json', 'netcon-semantic', 'Artefatto 05i-b non trovato.', EXPECTED_05IB_INDEX_SHA256)
ARTIFACT_05IC_SOURCE = artifact_source('HAYFLOW_05IC_ARTIFACT', 'hayflow_hines_synaptic_domain_repair.zip', 'synaptic_domain_repair_config.json', 'synaptic-domain', 'Artefatto 05i-c non trovato.', EXPECTED_05IC_INDEX_SHA256)
ARTIFACT_05J_SOURCE = artifact_source('HAYFLOW_05J_ARTIFACT', 'hayflow_hines_repaired_representation_recheck.zip', 'repaired_representation_recheck_config.json', 'repaired-representation-recheck', 'Artefatto 05j non trovato.', EXPECTED_05J_INDEX_SHA256)
ARTIFACT_05JB_SOURCE = artifact_source('HAYFLOW_05JB_ARTIFACT', 'hayflow_hines_repaired_representation_revision.zip', 'repaired_representation_revision_config.json', 'repaired-representation-revision', 'Artefatto 05j-b non trovato.', EXPECTED_05JB_INDEX_SHA256)
ARTIFACT_05JC_SOURCE = artifact_source('HAYFLOW_05JC_ARTIFACT', 'hayflow_hines_spatial_support_revision.zip', 'spatial_support_revision_config.json', 'spatial-support', 'Artefatto 05j-c non trovato.', EXPECTED_05JC_INDEX_SHA256)
print({'manifest': str(COMPOSITE_MANIFEST), 'base': str(BASE_SOURCE), '05j-c': str(ARTIFACT_05JC_SOURCE)})

## 3. Composite dataset and cryptographic preflight

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started, hash_last = {}, {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now); percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9); eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 05j-d][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True); hash_last[name] = percent
bundle = prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST, base_source=BASE_SOURCE, progress=hash_progress)
display({'valid': bundle.manifest['valid'], 'fingerprint': bundle.fingerprint, 'transition_count': bundle.transition_count})
assert bundle.manifest['valid'] and bundle.transition_count == 29880 and not bundle.manifest['physical_merge_performed']

## 4. Session construction and sealed methodology

In [ ]:
from src.hayflow_model import HinesCapacityConfig, HinesConditioningConfig, HinesIsolationConfig, HinesNetConSemanticRepairConfig, HinesOptimizationAuditConfig, HinesPrototypeExperimentConfig, HinesRepairedRepresentationRecheckConfig, HinesRepairedRepresentationRevisionConfig, HinesRepresentationForensicsConfig, HinesSegmentCanaryConfig, HinesSpatialSupportRevisionConfig, HinesStateNormalizationRepairConfig, HinesSynapticDomainRepairConfig, HinesTrainableTopologyCanary, HinesTrainableTopologyCanaryConfig
base_config = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_optimization_audit.yml').read_text())
forensic_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_representation_forensics.yml').read_text())
repair_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_state_normalization_repair.yml').read_text())
netcon_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_netcon_semantic_repair.yml').read_text())
domain_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_synaptic_domain_repair.yml').read_text())
recheck_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_repaired_representation_recheck.yml').read_text())
revision_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_repaired_representation_revision.yml').read_text())
spatial_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_spatial_support_revision.yml').read_text())
topology_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_trainable_topology_canary.yml').read_text())
model_config = HinesPrototypeExperimentConfig.from_mapping(base_config['model_experiment']); isolation_config = HinesIsolationConfig.from_mapping(base_config['isolation']); conditioning_config = HinesConditioningConfig.from_mapping(base_config['conditioning']); capacity_config = HinesCapacityConfig.from_mapping(base_config['capacity']); canary_config = HinesSegmentCanaryConfig.from_mapping(base_config['micro_canary']); audit_config = HinesOptimizationAuditConfig.from_mapping(base_config['optimization_audit'])
representation_config = HinesRepresentationForensicsConfig.from_mapping(forensic_payload['representation_forensics']); repair_config = HinesStateNormalizationRepairConfig.from_mapping(repair_payload['state_normalization_repair']); netcon_config = HinesNetConSemanticRepairConfig.from_mapping(netcon_payload['netcon_semantic_repair']); domain_config = HinesSynapticDomainRepairConfig.from_mapping(domain_payload['synaptic_domain_repair']); recheck_config = HinesRepairedRepresentationRecheckConfig.from_mapping(recheck_payload['repaired_representation_recheck']); revision_config = HinesRepairedRepresentationRevisionConfig.from_mapping(revision_payload['repaired_representation_revision']); spatial_config = HinesSpatialSupportRevisionConfig.from_mapping(spatial_payload['spatial_support_revision']); topology_config = HinesTrainableTopologyCanaryConfig.from_mapping(topology_payload['trainable_topology_canary'])
OUTPUT_DIR = Path('/kaggle/working/artifacts/hayflow_hines_trainable_topology_decoder_micro_canary')
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
session = HinesTrainableTopologyCanary(bundle, OUTPUT_DIR, model_config, isolation_config, conditioning_config, capacity_config, canary_config, audit_config, representation_config, CHECKPOINT_05B_SOURCE, ARTIFACT_05C_SOURCE, ARTIFACT_05D_SOURCE, ARTIFACT_05E_SOURCE, ARTIFACT_05F_SOURCE, ARTIFACT_05G_SOURCE, repair_config=repair_config, artifact_05h_source=ARTIFACT_05H_SOURCE, netcon_config=netcon_config, artifact_05i_source=ARTIFACT_05I_SOURCE, domain_config=domain_config, artifact_05ib_source=ARTIFACT_05IB_SOURCE, recheck_config=recheck_config, artifact_05ic_source=ARTIFACT_05IC_SOURCE, revision_config=revision_config, artifact_05j_source=ARTIFACT_05J_SOURCE, spatial_config=spatial_config, artifact_05jb_source=ARTIFACT_05JB_SOURCE, topology_config=topology_config, artifact_05jc_source=ARTIFACT_05JC_SOURCE, code_revision=REVISION)
prepare_report = session.prepare_trainable_topology_canary()
display({'revision': REVISION, '05j-c': prepare_report['artifact_05jc'], 'selection_roles': prepare_report['selection_roles']})
assert not prepare_report['development_used_for_selection'] and not prepare_report['heldout_inputs_extracted'] and not prepare_report['rollout_performed'] and not prepare_report['full_training_authorized']

## 5. Rebuild frozen features; make the 36/12 split before fitting transforms

In [ ]:
normalizer_report = session.apply_verified_synaptic_domain_normalizer()
support_report = session.build_expanded_train_support()
feature_report = session.prepare_expanded_spatial_features()
design_report = session.prepare_topology_canary_designs()
display({k: design_report[k] for k in ['valid', 'fit_pair_count', 'calibration_pair_count', 'development_pair_count', 'feature_width', 'calibration_family_quotas', 'fit_calibration_episode_overlap']})
assert design_report['valid'] and not design_report['fit_calibration_episode_overlap']
assert design_report['normalization_fit_roles'] == ['fit']
assert not design_report['development_used_for_checkpoint_selection'] and not design_report['heldout_inputs_extracted']

## 6. Frozen linear reference selected by fit-only grouped CV

In [ ]:
ridge_report = session.fit_fixed_tree_ridge_baseline()
display({'lambda': ridge_report['selected_ridge_lambda'], 'cv': ridge_report['cross_validation']['aggregate_voltage_rmse_mv'], 'fit': ridge_report['roles']['fit']['aggregate_voltage_rmse_mv'], 'calibration': ridge_report['roles']['calibration']['aggregate_voltage_rmse_mv'], 'development': ridge_report['roles']['development']['aggregate_voltage_rmse_mv']})
assert not ridge_report['calibration_used_for_selection'] and not ridge_report['development_used_for_selection']

## 7. Two-family, three-seed trainable topology micro-canary

In [ ]:
canary_report = session.run_trainable_topology_canary()
display(pd.DataFrame(canary_report['family_summary']))
assert canary_report['valid']
assert not canary_report['development_used_for_checkpoint_selection']
assert canary_report['development_inference_after_checkpoint_freeze']
assert not canary_report['heldout_inputs_extracted'] and not canary_report['rollout_performed']

## 8. Scoped decision — this never authorizes full training

In [ ]:
final_report = session.finalize_trainable_topology_canary(design_report, ridge_report, canary_report)
display({'valid': final_report['valid'], 'diagnosis': final_report['diagnosis'], 'passed': final_report['trainable_topology_canary_passed'], 'passing_families': final_report['passing_families'], 'improvement': final_report['improvement_vs_fixed_ridge'], 'next_step': final_report['next_step']})
assert final_report['valid'] and not final_report['full_training_authorized']
assert not final_report['heldout_contract']['inputs_extracted']
assert not final_report['methodology']['rollout_performed']
assert not final_report['methodology']['development_used_for_checkpoint_selection']

## 9. Create and download the ZIP

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, display
zip_base = Path('/kaggle/working/hayflow_hines_trainable_topology_decoder_micro_canary')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
payload = base64.b64encode(zip_path.read_bytes()).decode('ascii'); filename = zip_path.name
display(Javascript(f"""
const binary = atob('{payload}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
"""))
print({'zip': str(zip_path), 'size_mib': round(zip_path.stat().st_size / 2**20, 2), 'download': 'avviato dal browser'})